# Análisis de patrones por intención — corpus shiwilu

Caracteriza cada intención (SAL, EMO, PRG, REQUEST, AFI, NEG, DES) por sus **patrones**:
palabras y terminaciones características, marcadores documentados en la literatura, longitud
y riqueza léxica, y posibles atajos superficiales. El análisis se hace **solo sobre `train`**
(la partición evita fuga) para que sirva de base al **diseño del aumento de datos** (§10).


## 0. Parámetros, configuración y carga

In [1]:
import re
import sys
import math
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 180)

# ----------------------------- PARÁMETROS ----------------------------------- #
RANDOM_STATE = 42                       # semilla única para todo el notebook
PROP_TRAIN, PROP_DEV, PROP_TEST = 0.70, 0.15, 0.15   # deben sumar 1.0

MIN_DOCUMENT_FREQUENCY = 3    # nº mínimo de ENUNCIADOS (en train) con el patrón
MIN_COVERAGE           = 0.05 # cobertura mínima dentro de la categoría (5 %)
LOGODDS_ALPHA          = 0.5  # suavizado Haldane-Anscombe del log-odds
SEQ_LENGTHS            = [2, 3, 4, 5, 6]  # longitudes de secuencia inicial/final
NGRAM_ORDERS           = [1, 2, 3]        # n-gramas de palabras
TOP_K                  = 15   # nº de candidatos a mostrar por categoría
MATTR_WINDOW           = 25   # ventana (en tokens) para el MATTR
LOGODDS_FUERTE         = 1.0  # umbral de log-odds para "coincidencia directa"

# Nombres de columnas esperados en el CSV
COL_ES, COL_SHI, COL_LAB = "espanol", "shiwilu", "intencion"

# Categorías canónicas de Tesis 1. NO se modifica ninguna etiqueta del dataset:
# esto solo fija un ORDEN de presentación y descripciones de respaldo.
INTENCIONES_CANON = ["SAL", "EMO", "PRG", "REQUEST", "AFI", "NEG", "DES"]

DESC_INT_FALLBACK = {
    "SAL": "Saludos y despedidas",
    "EMO": "Expresiones emocionales",
    "PRG": "Preguntas informativas",
    "REQUEST": "Solicitudes y mandatos",
    "AFI": "Afirmaciones",
    "NEG": "Negaciones",
    "DES": "Descripciones de estados o eventos",
}
TIPO_SEARLE_FALLBACK = {
    "SAL": "Expresivo", "EMO": "Expresivo",
    "PRG": "Directivo", "REQUEST": "Directivo",
    "AFI": "Asertivo", "NEG": "Asertivo", "DES": "Asertivo",
}

assert abs(PROP_TRAIN + PROP_DEV + PROP_TEST - 1.0) < 1e-9, "Las proporciones deben sumar 1.0"
print("Parámetros cargados. random_state =", RANDOM_STATE)


Parámetros cargados. random_state = 42


In [2]:
# Raiz del repositorio: se busca hacia arriba el pyproject.toml, de modo que el
# notebook funcione sin importar desde donde se haya lanzado Jupyter.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

# Taxonomia y reglas de anotacion de la Fase 1 (nucleo compartido `shiwilu/`).
# La Seccion 10 usa anotar_intencion() como baseline por reglas frente a los
# clasificadores basados en embeddings.
#
# El import es directo y sin respaldo a proposito: si falla, es preferible
# detener el analisis a ejecutarlo contra una taxonomia distinta de la que se
# uso para construir el corpus.
from shiwilu.anotacion import anotar_intencion
from shiwilu.taxonomia import DESC_INT as _DESC, TIPO_SEARLE as _TIPO

DESC_INT = dict(_DESC)
TIPO_SEARLE = dict(_TIPO)
CONFIG_OK = True
print("Taxonomia y reglas importadas desde el paquete `shiwilu`.")

Taxonomia y reglas importadas desde el paquete `shiwilu`.


In [3]:
# Carga del corpus (producto de la Fase 1) y destino de las tablas de salida.
from shiwilu.rutas import CORPUS_CSV, TABLAS

RUTA_CORPUS = CORPUS_CSV
RUTA_SALIDA = TABLAS

if not RUTA_CORPUS.exists():
    raise FileNotFoundError(f"No se encontró el archivo del corpus: {RUTA_CORPUS}")


if not RUTA_SALIDA.exists():
    RUTA_SALIDA.mkdir(parents=True, exist_ok=True)


df = pd.read_csv(RUTA_CORPUS)

# Normalización tolerante de nombres de columna
_norm = {c.lower().strip(): c for c in df.columns}
_alias = {COL_ES: ["espanol", "español", "es", "texto_es"],
          COL_SHI: ["shiwilu", "shi", "texto_shi", "target"],
          COL_LAB: ["intencion", "intención", "label", "etiqueta", "categoria", "categoría"]}
for destino, posibles in _alias.items():
    for p in posibles:
        if p in _norm and _norm[p] != destino:
            df = df.rename(columns={_norm[p]: destino})
            break

faltan = [c for c in (COL_ES, COL_SHI, COL_LAB) if c not in df.columns]
if faltan:
    raise KeyError(f"Faltan columnas esperadas en el CSV: {faltan}. Columnas presentes: {list(df.columns)}")

print(f"Corpus cargado desde: {RUTA_CORPUS}")
print(f"Filas: {len(df)} | Columnas: {list(df.columns)}")
df[[COL_ES, COL_SHI, COL_LAB]].head()


Corpus cargado desde: C:\Lhia\Cursos Catolica\2026\TESIS\repo tesis\shiwilu-tesis\corpus\corpus_shiwilu_final.csv
Filas: 700 | Columnas: ['id', 'espanol', 'shiwilu', 'intencion', 'fuente']


,espanol,shiwilu,intencion
0,Claro.,tekinchi,AFI
1,Exacto.,nanapi'la,AFI
2,Correcto.,Musu',AFI
3,Verdad.,tekinchi,AFI
4,Perfecto.,mupalli,AFI


In [4]:
# Fijar el orden de INTENCIONES a partir de las etiquetas REALES del dataset
# y reportar cualquier inconsistencia respecto de las 7 categorías canónicas.
labels_data = list(pd.Series(df[COL_LAB].dropna().unique()))
INTENCIONES = [c for c in INTENCIONES_CANON if c in labels_data]
extra = [l for l in labels_data if l not in INTENCIONES_CANON]
INTENCIONES += sorted(extra)

print(f"Categorías encontradas en el dataset: {len(labels_data)}")
print(f"Orden de análisis (INTENCIONES): {INTENCIONES}")
if len(INTENCIONES) != 7:
    print(f"\n[REPORTE] El dataset tiene {len(INTENCIONES)} categorías, no 7.")
if extra:
    print(f"[REPORTE] Etiquetas fuera de la taxonomía canónica de 7: {extra}")
    print("          (No se fusionan ni renombran; solo se reportan.)")
faltantes_canon = [c for c in INTENCIONES_CANON if c not in labels_data]
if faltantes_canon:
    print(f"[REPORTE] Categorías canónicas ausentes en los datos: {faltantes_canon}")

# Asegurar descripciones/tipos para toda etiqueta presente
for l in INTENCIONES:
    DESC_INT.setdefault(l, l)
    TIPO_SEARLE.setdefault(l, "—")


Categorías encontradas en el dataset: 7
Orden de análisis (INTENCIONES): ['SAL', 'EMO', 'PRG', 'REQUEST', 'AFI', 'NEG', 'DES']


## 1. Panorama del corpus

Distribución por intención y duplicados. Los duplicados son relevantes para el aumento:
varias oraciones en español se tradujeron al mismo shiwilu.


In [5]:
print("="*64)
print("A.1  Tamaño y distribución de etiquetas (corpus completo)")
print("="*64)
print(f"Total de registros: {len(df)}")
print(f"Categorías: {df[COL_LAB].nunique()}")

resumen_corpus = (
    df[COL_LAB].value_counts()
      .reindex(INTENCIONES)
      .rename("n_enunciados")
      .to_frame()
)
resumen_corpus["porcentaje"] = (100 * resumen_corpus["n_enunciados"] / len(df)).round(1)
_desb = resumen_corpus["n_enunciados"].max() / max(resumen_corpus["n_enunciados"].min(), 1)
print(f"Razón de desbalance (mayor/menor): {_desb:.1f}x")
resumen_corpus


A.1  Tamaño y distribución de etiquetas (corpus completo)
Total de registros: 700
Categorías: 7
Razón de desbalance (mayor/menor): 1.0x


,n_enunciados,porcentaje
intencion,,
SAL,100,14.3
EMO,100,14.3
PRG,100,14.3
REQUEST,100,14.3
AFI,100,14.3
NEG,100,14.3
DES,100,14.3


In [6]:
print("="*64)
print("A.3  Duplicados e inconsistencias (se reportan, NO se eliminan)")
print("="*64)

dup_shi = df[df.duplicated(subset=[COL_SHI], keep=False)].sort_values(COL_SHI)
dup_es  = df[df.duplicated(subset=[COL_ES], keep=False)].sort_values(COL_ES)
print(f"Enunciados shiwilu duplicados (filas): {len(dup_shi)}")
print(f"Traducciones españolas duplicadas (filas): {len(dup_es)}")

# Mismo enunciado shiwilu con etiquetas distintas -> inconsistencia de anotación
conflicto = (df.groupby(COL_SHI)[COL_LAB].nunique()
               .loc[lambda s: s > 1])
print(f"Enunciados shiwilu con MÁS de una etiqueta: {conflicto.size}")
if conflicto.size:
    print("  (posible inconsistencia de anotación / fuga si caen en splits distintos)")
    display(df[df[COL_SHI].isin(conflicto.index)]
              .sort_values(COL_SHI)[[COL_SHI, COL_ES, COL_LAB]].head(20))

# Par (shiwilu, intencion) exactamente repetido
dup_par = df.duplicated(subset=[COL_SHI, COL_LAB]).sum()
print(f"Pares (shiwilu, intencion) exactamente repetidos: {dup_par}")


A.3  Duplicados e inconsistencias (se reportan, NO se eliminan)
Enunciados shiwilu duplicados (filas): 52
Traducciones españolas duplicadas (filas): 0
Enunciados shiwilu con MÁS de una etiqueta: 0
Pares (shiwilu, intencion) exactamente repetidos: 28


## 2. Partición `train` / `dev` / `test` (sin fuga)

Partición **agrupada por texto shiwilu**: las copias de un mismo enunciado caen en el mismo
split. Todos los patrones se calculan solo en `train`.


In [7]:
from sklearn.model_selection import train_test_split

_split_cols = [c for c in df.columns
               if c.lower().strip() in {"split", "particion", "partición", "conjunto", "fold"}]

if _split_cols:
    SPLIT_COL = _split_cols[0]
    print(f"Se encontró una partición oficial en la columna '{SPLIT_COL}'. Se respeta.")
    df["_split"] = df[SPLIT_COL].astype(str).str.lower().str.strip()
    _mapa = {"training": "train", "entrenamiento": "train", "tr": "train",
             "validation": "dev", "val": "dev", "validacion": "dev", "validación": "dev",
             "testing": "test", "te": "test", "prueba": "test"}
    df["_split"] = df["_split"].replace(_mapa)
else:
    print("No hay partición oficial. Se crea una estratificada y reproducible,")
    print("AGRUPANDO por texto shiwilu para que copias idénticas NO crucen splits.")
    # Cada texto shiwilu tiene UNA sola etiqueta (verificado en A.3), así que
    # partimos a nivel de TIPO (texto único), no de fila, y luego mapeamos.
    tipos = df[[COL_SHI, COL_LAB]].drop_duplicates(COL_SHI).reset_index(drop=True)
    t_idx = tipos.index.to_numpy()
    t_y = tipos[COL_LAB].to_numpy()
    rel_test = PROP_TEST / (PROP_DEV + PROP_TEST)
    def _split_seguro(indices, y_strat, test_size):
        # Estratifica por etiqueta; si alguna clase tiene <2 tipos, cae a split simple.
        try:
            return train_test_split(indices, test_size=test_size,
                                    random_state=RANDOM_STATE, stratify=y_strat)
        except ValueError:
            print("  [aviso] muy pocos textos únicos en alguna categoría; "
                  "split agrupado SIN estratificar (se mantiene sin fuga).")
            return train_test_split(indices, test_size=test_size,
                                    random_state=RANDOM_STATE, stratify=None)
    ti_train, ti_tmp = _split_seguro(t_idx, t_y, PROP_DEV + PROP_TEST)
    ti_dev, ti_test = _split_seguro(ti_tmp, tipos.loc[ti_tmp, COL_LAB].to_numpy(), rel_test)
    split_de_tipo = {}
    for _i in ti_train: split_de_tipo[tipos.loc[_i, COL_SHI]] = "train"
    for _i in ti_dev:   split_de_tipo[tipos.loc[_i, COL_SHI]] = "dev"
    for _i in ti_test:  split_de_tipo[tipos.loc[_i, COL_SHI]] = "test"
    df["_split"] = df[COL_SHI].map(split_de_tipo)

df_train = df[df["_split"] == "train"].copy()
df_dev   = df[df["_split"] == "dev"].copy()
df_test  = df[df["_split"] == "test"].copy()

print(f"\ntrain={len(df_train)}  dev={len(df_dev)}  test={len(df_test)}")
dist_split = pd.crosstab(df[COL_LAB], df["_split"]).reindex(INTENCIONES)[["train", "dev", "test"]]
print("\nDistribución por intención y split:")
dist_split


No hay partición oficial. Se crea una estratificada y reproducible,
AGRUPANDO por texto shiwilu para que copias idénticas NO crucen splits.

train=494  dev=102  test=104

Distribución por intención y split:


_split,train,dev,test
intencion,,,
SAL,71,14,15
EMO,70,15,15
PRG,71,14,15
REQUEST,70,15,15
AFI,71,15,14
NEG,70,15,15
DES,71,14,15


In [8]:
print("="*64)
print("B.1  Control de fuga entre splits (duplicados que cruzan train/dev/test)")
print("="*64)
_por_texto = df.groupby(COL_SHI)["_split"].agg(lambda s: set(s))
_cruzan = _por_texto[_por_texto.apply(len) > 1]
print(f"Enunciados shiwilu presentes en más de un split: {_cruzan.size}")
if _cruzan.size:
    print("  [ALERTA] Hay solapamiento de texto entre splits -> revisar antes de entrenar.")
    display(df[df[COL_SHI].isin(_cruzan.index)]
              .sort_values(COL_SHI)[[COL_SHI, COL_LAB, "_split"]].head(20))
else:
    print("  Sin solapamiento de texto entre splits.")


B.1  Control de fuga entre splits (duplicados que cruzan train/dev/test)
Enunciados shiwilu presentes en más de un split: 0
  Sin solapamiento de texto entre splits.


## 3. Utilidades de texto

Tokenización que conserva el apóstrofo interno y separa la puntuación; grafemas con dígrafos
`ch`/`ll`/`sh`; y asociación patrón↔intención por **cobertura de enunciados** con **log-odds
suavizado** (Haldane-Anscombe), que mide concentración sin premiar lo meramente frecuente.


In [9]:
PUNCT_BORDE = "¿?¡!.,;:\"()[]{}«»…—–\u2018\u2019\u201c\u201d"
DIGRAFOS = ("ch", "ll", "sh")


def tokenizar(texto):
    """Tokenización léxica: quita puntuación de los bordes, conserva el apóstrofo interno."""
    toks = []
    for bruto in str(texto).split():
        t = bruto.strip(PUNCT_BORDE).lower()
        # quitar apóstrofos SOLO si quedaron en un borde sin ninguna letra alrededor
        if any(ch.isalpha() for ch in t):
            toks.append(t)
    return toks


def features_ortograficas(texto):
    """Presencia de signos de puntuación (analizados aparte de las palabras)."""
    s = str(texto)
    return {
        "tiene_¿": "¿" in s,
        "tiene_?": "?" in s,
        "tiene_¡": "¡" in s,
        "tiene_!": "!" in s,
    }


def segmentar_grafemas(palabra):
    """Segmenta una palabra en grafemas del shiwilu.

    Reconoce primero los dígrafos ch, ll, sh; el apóstrofo ' se trata como grafema
    propio. NOTA: la rótica glotalizada r' y la consonante compleja ku (R4 §2) NO se
    fusionan aquí para no sobre-especificar; quedan como grafemas simples contiguos.
    """
    p = str(palabra).lower()
    out, i = [], 0
    while i < len(p):
        if p[i:i + 2] in DIGRAFOS:
            out.append(p[i:i + 2]); i += 2
        else:
            out.append(p[i]); i += 1
    return out


def ngramas(seq, n):
    """n-gramas de una secuencia (lista), SIN cruzar fronteras (seq es un enunciado)."""
    return [tuple(seq[i:i + n]) for i in range(len(seq) - n + 1)]


def _riesgo_shortcut(cov_cat, cov_resto):
    if cov_cat >= 0.50 and cov_resto <= 0.05: return "ALTO"
    if cov_cat >= 0.30 and cov_resto <= 0.10: return "MEDIO"
    if cov_cat >= 0.15 and cov_resto <= 0.15: return "BAJO"
    return "insignificante"


def tabla_asociacion(sub_df, extractor, labels, col=COL_SHI,
                     min_df=MIN_DOCUMENT_FREQUENCY, alpha=LOGODDS_ALPHA):
    """Construye una tabla (patrón x intención) con cobertura y log-odds por ENUNCIADO.

    extractor(text) -> conjunto de patrones candidatos presentes en el enunciado.
    Se usa presencia por enunciado (set): un patrón repetido dentro del mismo
    enunciado NO infla la evidencia.
    """
    n_por_label = sub_df[COL_LAB].value_counts().to_dict()
    N = int(sum(n_por_label.get(l, 0) for l in labels))
    cand_label = defaultdict(Counter)   # patrón -> Counter(label -> nº enunciados)
    for texto, lab in zip(sub_df[col], sub_df[COL_LAB]):
        for c in set(extractor(texto)):
            cand_label[c][lab] += 1

    filas = []
    for cand, lc in cand_label.items():
        total_docs = int(sum(lc.values()))
        if total_docs < min_df:
            continue
        for lab in labels:
            a = int(lc.get(lab, 0))
            if a == 0:
                continue
            n1 = int(n_por_label.get(lab, 0))
            b = total_docs - a
            n2 = N - n1
            cov_cat = a / n1 if n1 else 0.0
            cov_rest = b / n2 if n2 else 0.0
            lo = (math.log((a + alpha) / (n1 - a + alpha))
                  - math.log((b + alpha) / (n2 - b + alpha)))
            filas.append({
                "patron": cand, "intencion": lab,
                "df_cat": a, "n_cat": n1, "cobertura_cat": round(cov_cat, 4),
                "df_resto": b, "cobertura_resto": round(cov_rest, 4),
                "especificidad": round(a / total_docs, 4),
                "log_odds": round(lo, 3),
            })
    return pd.DataFrame(filas)


def mejores_por_categoria(asoc_df, labels, top_k=TOP_K,
                          min_df=MIN_DOCUMENT_FREQUENCY, min_cov=MIN_COVERAGE):
    """De una tabla de asociación devuelve (con_evidencia, raros).

    - con_evidencia: cumple df_cat>=min_df y cobertura_cat>=min_cov; top_k por log_odds
      dentro de cada categoría.
    - raros: candidatos que NO alcanzan los umbrales (se conservan, no se borran).
    """
    if asoc_df.empty:
        return asoc_df.copy(), asoc_df.copy()
    fuerte = asoc_df[(asoc_df["df_cat"] >= min_df) & (asoc_df["cobertura_cat"] >= min_cov)]
    raros = asoc_df.drop(fuerte.index)
    partes = []
    for lab in labels:
        partes.append(fuerte[fuerte["intencion"] == lab]
                      .sort_values("log_odds", ascending=False).head(top_k))
    con_evidencia = pd.concat(partes, ignore_index=True) if partes else fuerte
    return con_evidencia, raros.sort_values("log_odds", ascending=False)


print("Utilidades definidas. Prueba rápida:")
print("  tokenizar('¿Kenma ma\\'nen?') ->", tokenizar("¿Kenma ma'nen?"))
print("  segmentar_grafemas('uk\\'etchu') ->", segmentar_grafemas("uk'etchu"))
print("  segmentar_grafemas('lli\\'api\\'nek') ->", segmentar_grafemas("lli'api'nek"))


Utilidades definidas. Prueba rápida:
  tokenizar('¿Kenma ma\'nen?') -> ['kenma', "ma'nen"]
  segmentar_grafemas('uk\'etchu') -> ['u', 'k', "'", 'e', 't', 'ch', 'u']
  segmentar_grafemas('lli\'api\'nek') -> ['ll', 'i', "'", 'a', 'p', 'i', "'", 'n', 'e', 'k']


## 4. Longitud y riqueza léxica por intención

Tamaño típico de los enunciados y variedad léxica (TTR, MATTR con ventana, hapax, vocabulario
exclusivo/compartido) de cada categoría. La riqueza léxica es además la **línea base** contra
la que se juzgará si el aumento aporta variedad real.


In [10]:
# Longitud sobre TRAIN
tok_train = {lab: [tokenizar(t) for t in df_train.loc[df_train[COL_LAB] == lab, COL_SHI]]
             for lab in INTENCIONES}

filas_long = []
for lab in INTENCIONES:
    n_pal = np.array([len(t) for t in tok_train[lab]])
    n_car = np.array([len(str(s)) for s in df_train.loc[df_train[COL_LAB] == lab, COL_SHI]])
    if n_pal.size == 0:
        continue
    filas_long.append({
        "intencion": lab,
        "n_enunciados": int(n_pal.size),
        "media_palabras": round(float(n_pal.mean()), 2),
        "mediana_palabras": float(np.median(n_pal)),
        "std_palabras": round(float(n_pal.std(ddof=0)), 2),
        "min_palabras": int(n_pal.min()),
        "max_palabras": int(n_pal.max()),
        "media_caracteres": round(float(n_car.mean()), 2),
    })
resumen_longitud = pd.DataFrame(filas_long).set_index("intencion").reindex(INTENCIONES)
resumen_longitud


,n_enunciados,media_palabras,mediana_palabras,std_palabras,min_palabras,max_palabras,media_caracteres
intencion,,,,,,,
SAL,71,1.69,2.0,0.46,1,2,15.59
EMO,70,1.77,2.0,0.56,1,3,18.04
PRG,71,1.55,2.0,0.50,1,2,15.99
REQUEST,70,1.59,2.0,0.49,1,2,14.83
AFI,71,1.85,2.0,0.60,1,3,13.76
NEG,70,1.63,2.0,0.54,1,3,15.33
DES,71,1.62,2.0,0.57,1,3,15.38


In [11]:
def mattr(tokens_lista, ventana=MATTR_WINDOW):
    """MATTR: media del TTR sobre ventanas deslizantes de `ventana` tokens.

    Se aplana la lista de enunciados en una secuencia de tokens de la categoría.
    Si hay menos tokens que la ventana, se devuelve el TTR global (con aviso implícito).
    """
    toks = [t for enun in tokens_lista for t in enun]
    if len(toks) < ventana:
        return (len(set(toks)) / len(toks)) if toks else np.nan
    ratios = []
    for i in range(len(toks) - ventana + 1):
        v = toks[i:i + ventana]
        ratios.append(len(set(v)) / ventana)
    return float(np.mean(ratios))


# Vocabulario por categoría (tipos) para exclusivo/compartido
vocab_por_cat = {lab: set(t for enun in tok_train[lab] for t in enun) for lab in INTENCIONES}

filas_riq = []
for lab in INTENCIONES:
    toks = [t for enun in tok_train[lab] for t in enun]
    tipos = set(toks)
    frec = Counter(toks)
    hapax = [w for w, c in frec.items() if c == 1]
    otros = set().union(*[vocab_por_cat[o] for o in INTENCIONES if o != lab]) if len(INTENCIONES) > 1 else set()
    exclusivo = tipos - otros
    compartido = tipos & otros
    filas_riq.append({
        "intencion": lab,
        "tokens": len(toks),
        "tipos": len(tipos),
        "TTR": round(len(tipos) / len(toks), 4) if toks else np.nan,
        "MATTR": round(mattr(tok_train[lab]), 4),
        "hapax": len(hapax),
        "prop_hapax": round(len(hapax) / len(tipos), 4) if tipos else np.nan,
        "vocab_exclusivo": len(exclusivo),
        "vocab_compartido": len(compartido),
    })
riqueza_lexica = pd.DataFrame(filas_riq).set_index("intencion").reindex(INTENCIONES)
print(f"(MATTR con ventana = {MATTR_WINDOW} tokens)")
riqueza_lexica


(MATTR con ventana = 25 tokens)


,tokens,tipos,TTR,MATTR,hapax,prop_hapax,vocab_exclusivo,vocab_compartido
intencion,,,,,,,,
SAL,120,65,0.5417,0.7092,44,0.6769,56,9
EMO,124,78,0.6290,0.8372,61,0.7821,67,11
PRG,110,85,0.7727,0.8577,72,0.8471,70,15
REQUEST,111,94,0.8468,0.8777,86,0.9149,79,15
AFI,131,59,0.4504,0.6336,47,0.7966,44,15
NEG,114,73,0.6404,0.7671,57,0.7808,64,9
DES,115,92,0.8000,0.8567,80,0.8696,77,15


## 5. Palabras características por intención (keyness)

Palabras cuya presencia se concentra en una categoría (alta cobertura dentro, baja fuera).
Es la vista central de “qué distingue a cada intención” a nivel de palabra.


In [12]:
asoc_palabras = tabla_asociacion(df_train, lambda t: tokenizar(t), INTENCIONES)

# frecuencia absoluta (token-level) por categoría para enriquecer la tabla
_frec_tok_cat = {lab: Counter(t for enun in tok_train[lab] for t in enun) for lab in INTENCIONES}
def _fabs(row):
    return int(_frec_tok_cat[row["intencion"]].get(row["patron"], 0))
if not asoc_palabras.empty:
    asoc_palabras["frec_abs_cat"] = asoc_palabras.apply(_fabs, axis=1)

palabras_caracteristicas, palabras_raras = mejores_por_categoria(asoc_palabras, INTENCIONES)
print(f"Palabras con evidencia suficiente: {len(palabras_caracteristicas)} | "
      f"candidatos raros (bajo umbral): {len(palabras_raras)}")
palabras_caracteristicas.head(TOP_K * 2)


Palabras con evidencia suficiente: 34 | candidatos raros (bajo umbral): 64


,patron,intencion,df_cat,n_cat,cobertura_cat,df_resto,cobertura_resto,especificidad,log_odds,frec_abs_cat
0,ma'pu'sin,SAL,9,71,0.1268,0,0.0000,1.0000,4.858,9
1,pante'chek,SAL,9,71,0.1268,0,0.0000,1.0000,4.858,9
2,inlli'ter,SAL,5,71,0.0704,0,0.0000,1.0000,4.249,5
3,awa,SAL,4,71,0.0563,0,0.0000,1.0000,4.034,4
4,muektu'kerker',SAL,4,71,0.0563,0,0.0000,1.0000,4.034,4
5,pa'echek,SAL,5,71,0.0704,1,0.0024,0.8333,3.148,5
6,musu',SAL,7,71,0.0986,15,0.0355,0.3182,1.120,7
7,enpu'ni,EMO,13,70,0.1857,0,0.0000,1.0000,5.295,13
8,enpu'nipa',EMO,5,70,0.0714,0,0.0000,1.0000,4.267,5
9,mushipa',EMO,5,70,0.0714,0,0.0000,1.0000,4.267,5


## 6. Terminaciones características (secuencias finales de grafemas)

Secuencias **finales** de grafemas (longitud 2–6) frecuentes por categoría, en **todas** las
palabras del enunciado. Son *candidatas subléxicas*, **no** morfemas confirmados (sin POS no
se distingue raíz de afijo); útiles para ver terminaciones típicas, p. ej. la de `i'ñi` (NEG).


In [13]:
def _seqs_finales(texto, L):
    out = set()
    for w in tokenizar(texto):
        g = segmentar_grafemas(w)
        if len(g) >= L:
            out.add("".join(g[-L:]))
    return out

_partes = []
for L in SEQ_LENGTHS:
    a = tabla_asociacion(df_train, lambda t, L=L: _seqs_finales(t, L), INTENCIONES)
    if not a.empty:
        a.insert(1, "longitud", L)
    _partes.append(a)
_fin_all = pd.concat(_partes, ignore_index=True) if _partes else pd.DataFrame()
secuencias_finales, _ = mejores_por_categoria(_fin_all, INTENCIONES)
print(f"Secuencias finales candidatas con evidencia: {len(secuencias_finales)}")
secuencias_finales.head(TOP_K)


Secuencias finales candidatas con evidencia: 105


,patron,longitud,intencion,df_cat,n_cat,cobertura_cat,df_resto,cobertura_resto,especificidad,log_odds
0,'sin,4,SAL,9,71,0.1268,0,0.0000,1.0000,4.858
1,te'chek,6,SAL,9,71,0.1268,0,0.0000,1.0000,4.858
2,pu'sin,6,SAL,9,71,0.1268,0,0.0000,1.0000,4.858
3,u'sin,5,SAL,9,71,0.1268,0,0.0000,1.0000,4.858
4,sin,3,SAL,9,71,0.1268,0,0.0000,1.0000,4.858
5,wa,2,SAL,6,71,0.0845,0,0.0000,1.0000,4.431
6,ik,2,SAL,5,71,0.0704,0,0.0000,1.0000,4.249
7,lli'ter,6,SAL,5,71,0.0704,0,0.0000,1.0000,4.249
8,walek,5,SAL,5,71,0.0704,0,0.0000,1.0000,4.249
9,i'ter,5,SAL,5,71,0.0704,0,0.0000,1.0000,4.249


## 7. Marcadores documentados y contraste con la literatura (ficha R4)

Búsqueda dirigida de las formas que la lingüística del shiwilu documenta (Valenzuela &
Gussenhoven 2013; Valenzuela 2012; Valenzuela et al. 2024) y su correspondencia con el corpus.
`documentacion` indica qué categorías tienen respaldo (NEG/PRG/REQUEST), cuáles parcial
(EMO/AFI) y cuáles ninguna (SAL/DES).


In [14]:
# Formas documentadas en R4 (ficha v4). Fuentes:
#   JIPA  = Valenzuela & Gussenhoven (2013), J. Int. Phonetic Assoc. 43(1)
#   Voces = Valenzuela (2012), Voces shiwilu, Fondo Editorial PUCP
#   Enc   = Valenzuela, Vásquez & Lomas (2024), Enciclopedia MinCul
# tipo_match: token | word_final | word_initial | substr
#
# Estado de documentación por categoría (ficha, tabla de cobertura):
DOC_STATUS = {
    "NEG": "documentada", "PRG": "documentada", "REQUEST": "documentada",
    "EMO": "parcial", "AFI": "parcial",
    "SAL": "sin documentación", "DES": "sin documentación",
}

MARCADORES_DOC = [
    # ---- NEG (documentada) ----
    {"forma": "i'n", "variantes": ["i'n"], "tipo": "raíz de negación", "match": "substr",
     "esperada": "NEG", "funcion": "Morfema de negación; se combina con el sufijo de persona (i'n+-lli -> i'ñi)",
     "fuente": "JIPA p.100; Voces p.65",
     "nota": "i'ñi = negación + 3SG (coalescencia n+l/ll -> n/ñ, Voces p.72), no morfema único."},
    {"forma": "-inpu'", "variantes": ["inpu'"], "tipo": "sufijo negativo (nominal)", "match": "substr",
     "esperada": "NEG", "funcion": "Sufijo negativo; se adhiere a nominales",
     "fuente": "Voces p.73",
     "nota": "Desplazamiento glotal al sufijarse (Voces p.73): la forma superficial puede variar."},

    # ---- PRG (documentada) ----
    {"forma": "a'cha", "variantes": ["a'cha", "'cha"], "tipo": "partícula interrogativa", "match": "word_final",
     "esperada": "PRG", "funcion": "Partícula interrogativa independiente; se liga y fusiona cuando la palabra previa termina en a",
     "fuente": "JIPA p.103; Voces p.77",
     "nota": "Puede aparecer separada (a'cha) o ligada por fusión (...'cha); por eso se busca por final de palabra."},
    {"forma": "a'ta'", "variantes": ["a'ta'"], "tipo": "partícula interrogativa (sorpresa)", "match": "token",
     "esperada": "PRG", "funcion": "Partícula interrogativa de sorpresa; mismo comportamiento prosódico que a'cha",
     "fuente": "JIPA p.103; Voces p.77", "nota": "También puede ligarse por fusión."},
    {"forma": "interrogativos léxicos", "match": "token",
     "variantes": ["ennupa", "enñupa'la", "ma'nen", "denken", "ma'pu'si'pa'", "ma'lusa'"],
     "tipo": "palabra interrogativa léxica", "esperada": "PRG",
     "funcion": "Palabras cuyo significado aporta la interrogación (dónde/de dónde/qué/quién/cómo/qué tipo)",
     "fuente": "Voces pp.63,65,77; JIPA pp.98,100,103",
     "nota": "No son partículas; el rasgo interrogativo es léxico."},

    # ---- REQUEST (documentada) ----
    {"forma": "-ker'/-e'r/-r'", "variantes": ["r'", "e'r", "ker", "wir"], "tipo": "imperativo 2SG", "match": "word_final",
     "esperada": "REQUEST",
     "funcion": "Imperativo de 2.ª persona singular; los textos lo regularizan terminado en r'",
     "fuente": "JIPA p.99; Voces pp.63-64,75",
     "nota": "Alternancia r'~r (pierde el rasgo glotal a final de palabra); -wir atestiguado (inpiwir). Formas sin apóstrofo no siempre se capturan."},
    {"forma": "enchuku'", "variantes": ["enchuku'"], "tipo": "cohortativo", "match": "token",
     "esperada": "REQUEST", "funcion": "Forma cohortativa ('vamos')",
     "fuente": "Voces p.67", "nota": "Cohortativo, no imperativo 2SG estricto."},

    # ---- EMO (parcial) ----
    {"forma": "chi", "variantes": ["chi"], "tipo": "partícula de pesar (masc.)", "match": "token",
     "esperada": "EMO", "funcion": "Pena o sorpresa del hablante (habla masculina); palabra independiente",
     "fuente": "JIPA pp.102-103; Voces p.76",
     "nota": "Solo como token; 'chi' también aparece como subcadena (no se cuenta ahí)."},
    {"forma": "ten", "variantes": ["ten"], "tipo": "partícula de pesar (fem.)", "match": "token",
     "esperada": "EMO", "funcion": "Equivalente femenino de chi (tono bajo)",
     "fuente": "JIPA p.103; Voces p.76", "nota": "Solo como token."},
    {"forma": "-sha", "variantes": ["sha"], "tipo": "diminutivo/afectivo", "match": "word_final",
     "esperada": "EMO", "funcion": "Sufijo diminutivo/afectivo; expresa afecto o atenuación",
     "fuente": "Voces pp.69-70",
     "nota": "Afectivo, no exclusivo de EMO; palatalización opcional (la'pisha -> lla'pisha)."},

    # ---- AFI (parcial) ----
    {"forma": "ajá", "variantes": ["ajá", "aja"], "tipo": "afirmación", "match": "token",
     "esperada": "AFI", "funcion": "Afirmación ('sí')", "fuente": "Voces pp.60-61,66", "nota": ""},
    {"forma": "ahã", "variantes": ["ahã", "aha"], "tipo": "interjección afirmativa", "match": "token",
     "esperada": "AFI", "funcion": "Interjección afirmativa",
     "fuente": "JIPA p.98", "nota": "Única aparición de la fricativa laríngea en el corpus del artículo."},
    {"forma": "untana", "variantes": ["untana"], "tipo": "afirmativo enfático", "match": "substr",
     "esperada": "AFI", "funcion": "Afirmativo enfático (acento inherente)", "fuente": "JIPA p.102", "nota": ""},
    {"forma": "i'na", "variantes": ["i'na"], "tipo": "focalizador", "match": "token",
     "esperada": "AFI", "funcion": "Focalizador; palabra independiente que se fusiona tras vocal",
     "fuente": "Voces pp.72,75",
     "nota": "Focalizador, no afirmación en sentido estricto; puede no mapear limpio a AFI."},
    {"forma": "-sa'", "variantes": ["sa'"], "tipo": "delimitativo ('solo')", "match": "word_final",
     "esperada": "AFI", "funcion": "Delimitativo ('solo'); marca foco estrecho",
     "fuente": "JIPA pp.102-103",
     "nota": "Delimitativo/foco, no afirmación; frecuente en varias categorías -> baja especificidad esperada."},

    # ---- Control metodológico (NO documentado en esta ficha) ----
    {"forma": "ya-", "variantes": ["ya"], "tipo": "prefijo desiderativo", "match": "word_initial",
     "esperada": None, "funcion": "Prefijo desiderativo (modalidad de deseo)",
     "fuente": "No figura en la ficha R4 v4; control metodológico",
     "nota": "NO asociar con DES=Descripción: 'desiderativo' es modalidad de deseo, no la categoría del corpus."},
]


def _presente(texto, variantes, match):
    ws = tokenizar(texto)
    for v in variantes:
        if match == "token" and v in ws:
            return True
        if match == "word_final" and any(w.endswith(v) for w in ws):
            return True
        if match == "word_initial" and any(w.startswith(v) for w in ws):
            return True
        if match == "substr" and any(v in w for w in ws):
            return True
    return False


def _cobertura_por_cat(mask_series, labels):
    """Devuelve dict label -> (df_cat, n_cat, cobertura)."""
    out = {}
    for lab in labels:
        idx = df_train[COL_LAB] == lab
        n1 = int(idx.sum())
        a = int((mask_series & idx).sum())
        out[lab] = (a, n1, a / n1 if n1 else 0.0)
    return out


filas_marc = []
N_train = len(df_train)
for m in MARCADORES_DOC:
    mask = df_train[COL_SHI].apply(lambda t: _presente(t, m["variantes"], m["match"]))
    cob = _cobertura_por_cat(mask, INTENCIONES)
    total = int(mask.sum())
    top_lab = max(INTENCIONES, key=lambda l: cob[l][2]) if total else None
    a, n1, cov_cat = cob[top_lab] if top_lab else (0, 0, 0.0)
    b = total - a
    n2 = N_train - n1
    cov_rest = b / n2 if n2 else 0.0
    lo = (math.log((a + LOGODDS_ALPHA) / (n1 - a + LOGODDS_ALPHA))
          - math.log((b + LOGODDS_ALPHA) / (n2 - b + LOGODDS_ALPHA))) if total else np.nan
    filas_marc.append({
        "forma_documentada": m["forma"], "tipo": m["tipo"], "match": m["match"],
        "intencion_esperada": m["esperada"] if m["esperada"] else "(n/a)",
        "documentacion": DOC_STATUS.get(m["esperada"], "(n/a)"),
        "df_total_train": total,
        "intencion_asociada_corpus": top_lab if total else "(no aparece)",
        "cobertura_categoria": round(cov_cat, 4),
        "cobertura_resto": round(cov_rest, 4),
        "log_odds": round(lo, 3) if total else np.nan,
        "funcion_documentada": m["funcion"], "fuente": m["fuente"], "nota": m["nota"],
    })
marcadores_documentados = pd.DataFrame(filas_marc)
marcadores_documentados

,forma_documentada,tipo,match,intencion_esperada,documentacion,df_total_train,intencion_asociada_corpus,cobertura_categoria,cobertura_resto,log_odds,funcion_documentada,fuente,nota
0,i'n,raíz de negación,substr,NEG,documentada,25,NEG,0.2286,0.0212,2.583,Morfema de negación; se combina con el sufijo de persona (i'n+-lli -> i'ñi),JIPA p.100; Voces p.65,"i'ñi = negación + 3SG (coalescencia n+l/ll -> n/ñ, Voces p.72), no morfema ú..."
1,-inpu',sufijo negativo (nominal),substr,NEG,documentada,8,NEG,0.1000,0.0024,3.507,Sufijo negativo; se adhiere a nominales,Voces p.73,Desplazamiento glotal al sufijarse (Voces p.73): la forma superficial puede ...
2,a'cha,partícula interrogativa,word_final,PRG,documentada,12,PRG,0.1690,0.0000,5.181,Partícula interrogativa independiente; se liga y fusiona cuando la palabra p...,JIPA p.103; Voces p.77,Puede aparecer separada (a'cha) o ligada por fusión (...'cha); por eso se bu...
3,a'ta',partícula interrogativa (sorpresa),token,PRG,documentada,0,(no aparece),0.0000,0.0000,NaN,Partícula interrogativa de sorpresa; mismo comportamiento prosódico que a'cha,JIPA p.103; Voces p.77,También puede ligarse por fusión.
4,interrogativos léxicos,palabra interrogativa léxica,token,PRG,documentada,0,(no aparece),0.0000,0.0000,NaN,Palabras cuyo significado aporta la interrogación (dónde/de dónde/qué/quién/...,"Voces pp.63,65,77; JIPA pp.98,100,103",No son partículas; el rasgo interrogativo es léxico.
5,-ker'/-e'r/-r',imperativo 2SG,word_final,REQUEST,documentada,46,REQUEST,0.3143,0.0566,2.026,Imperativo de 2.ª persona singular; los textos lo regularizan terminado en r',"JIPA p.99; Voces pp.63-64,75",Alternancia r'~r (pierde el rasgo glotal a final de palabra); -wir atestigua...
6,enchuku',cohortativo,token,REQUEST,documentada,0,(no aparece),0.0000,0.0000,NaN,Forma cohortativa ('vamos'),Voces p.67,"Cohortativo, no imperativo 2SG estricto."
7,chi,partícula de pesar (masc.),token,EMO,parcial,0,(no aparece),0.0000,0.0000,NaN,Pena o sorpresa del hablante (habla masculina); palabra independiente,JIPA pp.102-103; Voces p.76,Solo como token; 'chi' también aparece como subcadena (no se cuenta ahí).
8,ten,partícula de pesar (fem.),token,EMO,parcial,0,(no aparece),0.0000,0.0000,NaN,Equivalente femenino de chi (tono bajo),JIPA p.103; Voces p.76,Solo como token.
9,-sha,diminutivo/afectivo,word_final,EMO,parcial,8,NEG,0.0429,0.0118,1.375,Sufijo diminutivo/afectivo; expresa afecto o atenuación,Voces pp.69-70,"Afectivo, no exclusivo de EMO; palatalización opcional (la'pisha -> lla'pisha)."


In [15]:
def _interpretar(row, tiene_variantes):
    esperada = row["intencion_esperada"]
    total = row["df_total_train"]
    cov = row["cobertura_categoria"]
    lo = row["log_odds"]
    top = row["intencion_asociada_corpus"]
    if total < MIN_DOCUMENT_FREQUENCY or cov < MIN_COVERAGE or pd.isna(lo):
        return "No interpretable lingüísticamente con la evidencia disponible"
    if esperada == "(n/a)":
        return "Marcador documentado; no aplica asociación con categoría del corpus (p. ej. ya- desiderativo)"
    if top == esperada and lo >= LOGODDS_FUERTE:
        return "Coincidencia directa con literatura"
    if top == esperada:
        return ("Variante superficial posible" if tiene_variantes else "Coincidencia compatible")
    return "Patrón empírico no documentado en las fuentes revisadas (asociado a otra categoría)"


_tiene_var = {m["forma"]: (len(m["variantes"]) > 1) for m in MARCADORES_DOC}
contraste_corpus_literatura = marcadores_documentados.copy()
contraste_corpus_literatura["interpretacion"] = contraste_corpus_literatura.apply(
    lambda r: _interpretar(r, _tiene_var.get(r["forma_documentada"], False)), axis=1
)
contraste_corpus_literatura = contraste_corpus_literatura[[
    "forma_documentada", "tipo", "documentacion",
    "intencion_asociada_corpus", "intencion_esperada",
    "cobertura_categoria", "cobertura_resto", "log_odds",
    "funcion_documentada", "interpretacion", "nota",
]]

# Recordatorio del estado de documentación por categoría (ficha R4 v4)
print("Estado de documentación por categoría (ficha R4 v4):")
for cat in INTENCIONES:
    print(f"  {cat:<8} {DOC_STATUS.get(cat, '(n/a)')}")
print("\nSAL y DES no tienen marcadores en la literatura: los patrones que aparezcan")
print("para esas categorías se reportan como REGULARIDADES FORMALES del corpus,")
print("sin contraste bibliográfico. EMO y AFI: contraste solo PARCIAL.\n")
contraste_corpus_literatura

Estado de documentación por categoría (ficha R4 v4):
  SAL      sin documentación
  EMO      parcial
  PRG      documentada
  REQUEST  documentada
  AFI      parcial
  NEG      documentada
  DES      sin documentación

SAL y DES no tienen marcadores en la literatura: los patrones que aparezcan
para esas categorías se reportan como REGULARIDADES FORMALES del corpus,
sin contraste bibliográfico. EMO y AFI: contraste solo PARCIAL.



,forma_documentada,tipo,documentacion,intencion_asociada_corpus,intencion_esperada,cobertura_categoria,cobertura_resto,log_odds,funcion_documentada,interpretacion,nota
0,i'n,raíz de negación,documentada,NEG,NEG,0.2286,0.0212,2.583,Morfema de negación; se combina con el sufijo de persona (i'n+-lli -> i'ñi),Coincidencia directa con literatura,"i'ñi = negación + 3SG (coalescencia n+l/ll -> n/ñ, Voces p.72), no morfema ú..."
1,-inpu',sufijo negativo (nominal),documentada,NEG,NEG,0.1000,0.0024,3.507,Sufijo negativo; se adhiere a nominales,Coincidencia directa con literatura,Desplazamiento glotal al sufijarse (Voces p.73): la forma superficial puede ...
2,a'cha,partícula interrogativa,documentada,PRG,PRG,0.1690,0.0000,5.181,Partícula interrogativa independiente; se liga y fusiona cuando la palabra p...,Coincidencia directa con literatura,Puede aparecer separada (a'cha) o ligada por fusión (...'cha); por eso se bu...
3,a'ta',partícula interrogativa (sorpresa),documentada,(no aparece),PRG,0.0000,0.0000,NaN,Partícula interrogativa de sorpresa; mismo comportamiento prosódico que a'cha,No interpretable lingüísticamente con la evidencia disponible,También puede ligarse por fusión.
4,interrogativos léxicos,palabra interrogativa léxica,documentada,(no aparece),PRG,0.0000,0.0000,NaN,Palabras cuyo significado aporta la interrogación (dónde/de dónde/qué/quién/...,No interpretable lingüísticamente con la evidencia disponible,No son partículas; el rasgo interrogativo es léxico.
5,-ker'/-e'r/-r',imperativo 2SG,documentada,REQUEST,REQUEST,0.3143,0.0566,2.026,Imperativo de 2.ª persona singular; los textos lo regularizan terminado en r',Coincidencia directa con literatura,Alternancia r'~r (pierde el rasgo glotal a final de palabra); -wir atestigua...
6,enchuku',cohortativo,documentada,(no aparece),REQUEST,0.0000,0.0000,NaN,Forma cohortativa ('vamos'),No interpretable lingüísticamente con la evidencia disponible,"Cohortativo, no imperativo 2SG estricto."
7,chi,partícula de pesar (masc.),parcial,(no aparece),EMO,0.0000,0.0000,NaN,Pena o sorpresa del hablante (habla masculina); palabra independiente,No interpretable lingüísticamente con la evidencia disponible,Solo como token; 'chi' también aparece como subcadena (no se cuenta ahí).
8,ten,partícula de pesar (fem.),parcial,(no aparece),EMO,0.0000,0.0000,NaN,Equivalente femenino de chi (tono bajo),No interpretable lingüísticamente con la evidencia disponible,Solo como token.
9,-sha,diminutivo/afectivo,parcial,NEG,EMO,0.0429,0.0118,1.375,Sufijo diminutivo/afectivo; expresa afecto o atenuación,No interpretable lingüísticamente con la evidencia disponible,"Afectivo, no exclusivo de EMO; palatalización opcional (la'pisha -> lla'pisha)."


## 8. Posibles atajos ortográficos

Rasgos superficiales (sobre todo puntuación) que podrían predecir la intención sin aprender
shiwilu. `riesgo_shortcut` = ALTO cuando cubren casi toda una categoría y casi nada del resto.


In [16]:
# Stopwords españolas mínimas para detectar restos de español en la columna shiwilu
STOP_ES = {"que", "de", "la", "el", "los", "las", "un", "una", "y", "o", "no", "si",
           "sí", "es", "en", "con", "por", "para", "me", "te", "se", "su", "mi", "a",
           "como", "cómo", "qué", "quién", "dónde", "cuándo", "porque"}

filas_sc = []

def _add_shortcut(nombre, mask_bool):
    for lab in INTENCIONES:
        idx = df_train[COL_LAB] == lab
        n1 = int(idx.sum()); n2 = N_train - n1
        a = int((mask_bool & idx).sum()); b = int(mask_bool.sum()) - a
        cov_cat = a / n1 if n1 else 0.0
        cov_rest = b / n2 if n2 else 0.0
        if a == 0:
            continue
        filas_sc.append({
            "feature": nombre, "categoria": lab,
            "cobertura_categoria": round(cov_cat, 4),
            "cobertura_resto": round(cov_rest, 4),
            "riesgo_shortcut": _riesgo_shortcut(cov_cat, cov_rest),
        })

# signos de puntuación (sobre el texto CRUDO)
for sig, nombre in [("¿", "signo ¿"), ("?", "signo ?"), ("¡", "signo ¡"), ("!", "signo !")]:
    _add_shortcut(nombre, df_train[COL_SHI].str.contains(re.escape(sig), regex=True).fillna(False))

# restos de español
mask_es = df_train[COL_SHI].apply(lambda t: any(w in STOP_ES for w in tokenizar(t)))
_add_shortcut("token en español (posible resto)", mask_es)

# enunciados de 1 sola palabra (longitud extremadamente característica)
mask_1w = df_train[COL_SHI].apply(lambda t: len(tokenizar(t)) == 1)
_add_shortcut("enunciado de 1 palabra", mask_1w)

posibles_shortcuts = (pd.DataFrame(filas_sc)
                      .sort_values(["riesgo_shortcut", "cobertura_categoria"], ascending=[True, False])
                      if filas_sc else pd.DataFrame(
                          columns=["feature", "categoria", "cobertura_categoria",
                                   "cobertura_resto", "riesgo_shortcut"]))
# mostrar primero los de mayor riesgo
_orden = {"ALTO": 0, "MEDIO": 1, "BAJO": 2, "insignificante": 3}
if not posibles_shortcuts.empty:
    posibles_shortcuts = posibles_shortcuts.assign(
        _o=posibles_shortcuts["riesgo_shortcut"].map(_orden)
    ).sort_values(["_o", "cobertura_categoria"], ascending=[True, False]).drop(columns="_o")
posibles_shortcuts


,feature,categoria,cobertura_categoria,cobertura_resto,riesgo_shortcut
0,signo ¿,PRG,1.0000,0.0260,ALTO
2,signo ?,PRG,1.0000,0.0260,ALTO
5,signo ¡,EMO,0.1571,0.0377,BAJO
10,signo !,EMO,0.1571,0.0377,BAJO
16,enunciado de 1 palabra,PRG,0.4507,0.3522,insignificante
20,enunciado de 1 palabra,DES,0.4225,0.3570,insignificante
17,enunciado de 1 palabra,REQUEST,0.4143,0.3585,insignificante
19,enunciado de 1 palabra,NEG,0.4000,0.3608,insignificante
14,enunciado de 1 palabra,SAL,0.3099,0.3759,insignificante
15,enunciado de 1 palabra,EMO,0.3000,0.3774,insignificante


## 9. Principales hallazgos por intención

In [17]:
lineas = []
ap = lineas.append

# 1) Tamaño y desbalance
tot = int(resumen_corpus["n_enunciados"].sum())
cmax = int(resumen_corpus["n_enunciados"].max()); cmin = int(resumen_corpus["n_enunciados"].min())
ratio = cmax / max(cmin, 1)
if cmax == cmin:
    ap(f"- El corpus tiene {tot} enunciados repartidos de forma BALANCEADA en "
       f"{len(INTENCIONES)} categorías ({cmax} por categoría; desbalance {ratio:.1f}x). "
       f"Aun así se recomienda macro-F1 en la evaluación posterior.")
else:
    mayor = resumen_corpus["n_enunciados"].idxmax(); menor = resumen_corpus["n_enunciados"].idxmin()
    ap(f"- El corpus tiene {tot} enunciados en {len(INTENCIONES)} categorías. La más poblada "
       f"es {mayor} ({int(resumen_corpus.loc[mayor,'n_enunciados'])}) y la menos poblada "
       f"{menor} ({int(resumen_corpus.loc[menor,'n_enunciados'])}); "
       f"desbalance ~{ratio:.1f}x. Se recomienda macro-F1 en la evaluación posterior.")

# 2) Marcadores documentados con coincidencia
directas = contraste_corpus_literatura[
    contraste_corpus_literatura["interpretacion"].str.startswith(("Coincidencia", "Variante"))]
if len(directas):
    for _, r in directas.iterrows():
        ap(f"- La forma «{r['forma_documentada']}» aparece concentrada en "
           f"{r['intencion_asociada_corpus']} (cobertura {r['cobertura_categoria']:.0%} vs "
           f"{r['cobertura_resto']:.0%} en el resto, log-odds {r['log_odds']}): "
           f"{r['interpretacion'].lower()}.")
else:
    ap("- Con los umbrales actuales, ningún marcador documentado alcanza evidencia "
       "suficiente para una coincidencia directa; ver la tabla de contraste.")

# 3) Patrón subléxico final más asociado por categoría (top 1 por log-odds)
if not secuencias_finales.empty:
    top_fin = (secuencias_finales.sort_values("log_odds", ascending=False)
               .drop_duplicates("intencion").head(4))
    for _, r in top_fin.iterrows():
        ap(f"- Secuencia final candidata «{r['patron']}» (long. {int(r['longitud'])}) "
           f"concentrada en {r['intencion']} (cobertura {r['cobertura_cat']:.0%}, "
           f"log-odds {r['log_odds']}). Candidato subléxico, no morfema confirmado.")

# 4) Shortcuts de mayor riesgo
altos = posibles_shortcuts[posibles_shortcuts["riesgo_shortcut"].isin(["ALTO", "MEDIO"])]
if len(altos):
    for _, r in altos.head(4).iterrows():
        ap(f"- Posible atajo: «{r['feature']}» cubre {r['cobertura_categoria']:.0%} de "
           f"{r['categoria']} y {r['cobertura_resto']:.0%} del resto "
           f"(riesgo {r['riesgo_shortcut']}). Considerar experimentos con/sin ese rasgo.")
else:
    ap("- No se detectaron atajos ortográficos de riesgo ALTO/MEDIO con los umbrales actuales.")

print("PRINCIPALES HALLAZGOS (generados desde los resultados reales)\n" + "="*60)
print("\n".join(lineas))


PRINCIPALES HALLAZGOS (generados desde los resultados reales)
- El corpus tiene 700 enunciados repartidos de forma BALANCEADA en 7 categorías (100 por categoría; desbalance 1.0x). Aun así se recomienda macro-F1 en la evaluación posterior.
- La forma «i'n» aparece concentrada en NEG (cobertura 23% vs 2% en el resto, log-odds 2.583): coincidencia directa con literatura.
- La forma «-inpu'» aparece concentrada en NEG (cobertura 10% vs 0% en el resto, log-odds 3.507): coincidencia directa con literatura.
- La forma «a'cha» aparece concentrada en PRG (cobertura 17% vs 0% en el resto, log-odds 5.181): coincidencia directa con literatura.
- La forma «-ker'/-e'r/-r'» aparece concentrada en REQUEST (cobertura 31% vs 6% en el resto, log-odds 2.026): coincidencia directa con literatura.
- La forma «ajá» aparece concentrada en AFI (cobertura 31% vs 0% en el resto, log-odds 5.953): coincidencia directa con literatura.
- Secuencia final candidata «ajá» (long. 3) concentrada en AFI (cobertura 30%, lo

## 10. Qué aporta este análisis a la elección de la técnica de aumento

In [18]:
# Cifras que condicionan el aumento (desde los resultados reales).
conf = contraste_corpus_literatura[
    contraste_corpus_literatura["interpretacion"] == "Coincidencia directa con literatura"]
alto = posibles_shortcuts[posibles_shortcuts["riesgo_shortcut"] == "ALTO"]
print(f"Marcadores de señal confirmados (a preservar): {len(conf)} -> "
      + ", ".join(f"{r.forma_documentada}->{r.intencion_asociada_corpus}" for r in conf.itertuples()))
print(f"Rasgos de atajo ALTO (no depender de ellos): "
      + (", ".join(f"{r.feature}({r.categoria})" for r in alto.itertuples()) or "ninguno"))
print(f"Riqueza léxica base:  TTR medio={riqueza_lexica['TTR'].mean():.3f}  "
      f"MATTR medio={riqueza_lexica['MATTR'].mean():.3f}")
print(f"Duplicados: {len(dup_shi)} filas, {int(dup_par)} pares (shiwilu,intención) "
      f"-> deduplicar/agrupar antes de aumentar.")


Marcadores de señal confirmados (a preservar): 5 -> i'n->NEG, -inpu'->NEG, a'cha->PRG, -ker'/-e'r/-r'->REQUEST, ajá->AFI
Rasgos de atajo ALTO (no depender de ellos): signo ¿(PRG), signo ?(PRG)
Riqueza léxica base:  TTR medio=0.669  MATTR medio=0.791
Duplicados: 52 filas, 28 pares (shiwilu,intención) -> deduplicar/agrupar antes de aumentar.


Las técnicas candidatas se eligen por la literatura de bajos recursos; el corpus dicta **cómo**
aplicarlas y **qué vigilar**:

| Hallazgo del análisis | Implicación para el aumento |
|---|---|
| Marcadores de señal fuertes por categoría (§7) | Cada variante generada debe **conservar el marcador** de su intención; filtrar con un chequeo automático de supervivencia. |
| PRG se apoya en `¿?` + `a'cha` (§8) | El aumento **no debe depender de la puntuación**; cuidar que la retrotraducción no pierda `a'cha`. Evaluar con/sin `¿?`. |
| Duplicados de traducción (§1) | Aumentar **solo `train`** y **deduplicar/agrupar antes** de generar. |
| Corpus pequeño y balanceado (§1) | Volumen con incremento **≥100 % solo en train**; `dev`/`test` intactos. |
| Riqueza léxica base TTR/MATTR (§4) | Métrica de control: el aumento válido **sube la variedad**, no solo repite. |

**Lectura por técnica:**

- **Retrotraducción (Xie et al. 2020):** barata y efectiva para volumen; riesgo de *españolizar*
  y perder `a'cha`/marcadores → filtrar por marcador y revisar puntuación.
- **Generate-then-Refine (Lin et al. 2024):** la que más eleva la variedad léxica, pero puede
  generar shiwilu inexistente → el paso *refine* y la validación de una muestra son imprescindibles.
- **XL-LoRA (Basoz et al. 2026):** adaptación translingüe apta para corpus pequeño; mejora la
  representación más que generar texto nuevo → complementa a las anteriores.


## Limitaciones

- **Corpus escrito, sin información prosódica.** La literatura (R4 §5–§6, §8) indica que
  interrogación e imperativo se distinguen también por entonación; el corpus es escrito,
  así que esa distinción **no es recuperable** aquí.
- **Sin etiquetado morfológico / POS.** No hay anotación que permita identificar de forma
  fiable verbos, raíces o afijos; por eso **no se asume** que la última palabra sea el
  verbo ni se etiqueta ningún segmento como “sufijo”. Las secuencias son *candidatas*.
- **Corpus pequeño y desbalanceado.** Los tamaños por categoría limitan la potencia
  estadística; el log-odds usa suavizado, pero diferencias pequeñas no son concluyentes.
  El desbalance obliga a reportar macro-F1, no accuracy.
- **Patrones superficiales multi-análisis.** Una misma secuencia de grafemas puede
  corresponder a más de un análisis morfológico (p. ej. `i'ñi` = negación + 3SG, no un
  morfema único; el apóstrofo tiene más de una función).
- **Variación morfofonológica (fragmenta la frecuencia).** Un mismo morfema aparece con
  varias formas superficiales, por lo que su frecuencia real es mayor que la de cualquier
  forma aislada. Ejemplos documentados en la ficha: palatalización tras vibrante
  (`-lek`~`-llek`~`-tllek`, `-tek`~`-chek`, `-nen`~`-ñen`), coalescencia verbal
  (`n`+`l`/`ll` → `n`/`ñ`), geminación tras `e`, y alternancias libres (`-lli`~`-llin`,
  `we-`~`u-`, `r'`~`r`). La ausencia de una forma exacta **no** implica ausencia del morfema.
- **Dígrafo `ku` no segmentado.** La ficha documenta `ku` como grafía compuesta (consonante
  compleja). El segmentador de grafemas trata `ch`, `ll`, `sh` y el apóstrofo, pero **no**
  `ku`, porque su valor de dígrafo es fonológicamente condicionado; conviene tenerlo presente
  al leer las secuencias subléxicas.
- **PRG marcada por rasgos superficiales.** Las preguntas del corpus se construyen con el
  signo `¿?` y la partícula interrogativa `a'cha` (separada o fusionada, `…'cha`); los
  interrogativos léxicos documentados (`ma'nen`, `denken`, `ennupa`, `ma'lusa'`…) están
  **ausentes**, y algunas preguntas se marcan solo por entonación (`¿asek?`). Esto concentra
  la señal de PRG en rasgos superficiales y motiva evaluar el clasificador **con y sin**
  puntuación, para verificar que aprende la lengua y no el signo `¿?`.
- **SAL y DES sin respaldo bibliográfico.** La ficha no documenta marcadores para estas
  categorías; sus patrones se reportan como regularidades formales del corpus, no como
  coincidencias con la literatura. EMO y AFI solo admiten contraste parcial.
- **Un solo traductor/anotador.** No hay acuerdo interanotador; la validez descansa en la
  validación del hablante nativo (Etapa 1). Las inconsistencias de etiqueta detectadas en
  A.3 deben revisarse manualmente.
- **Oraciones base generadas con LLM en español** (Etapa 1): la validez ecológica se
  apoya en la validación del hablante, no en habla espontánea recolectada.
- **Diagnóstico en español (Sección 10) ≠ patrón oro.** El acuerdo con `anotar_intencion()`
  describe consistencia de anotación, no corrección lingüística del shiwilu.


In [19]:
for cat in ["SAL", "DES"]:
    print(f"\n{'='*50}\n{cat} — palabras características\n{'='*50}")
    print(palabras_caracteristicas[palabras_caracteristicas.intencion==cat]
          [["patron","cobertura_cat","cobertura_resto","especificidad","log_odds"]].to_string(index=False))
    print(f"\n{cat} — terminaciones finales")
    print(secuencias_finales[secuencias_finales.intencion==cat]
          [["patron","longitud","cobertura_cat","especificidad","log_odds"]].head(10).to_string(index=False))
    print(f"\n{cat} — candidatos bajo umbral (dispersión):",
          len(palabras_raras[palabras_raras.intencion==cat]))


SAL — palabras características
        patron  cobertura_cat  cobertura_resto  especificidad  log_odds
     ma'pu'sin         0.1268           0.0000         1.0000     4.858
    pante'chek         0.1268           0.0000         1.0000     4.858
     inlli'ter         0.0704           0.0000         1.0000     4.249
           awa         0.0563           0.0000         1.0000     4.034
muektu'kerker'         0.0563           0.0000         1.0000     4.034
      pa'echek         0.0704           0.0024         0.8333     3.148
         musu'         0.0986           0.0355         0.3182     1.120

SAL — terminaciones finales
 patron  longitud  cobertura_cat  especificidad  log_odds
   'sin         4         0.1268            1.0     4.858
te'chek         6         0.1268            1.0     4.858
 pu'sin         6         0.1268            1.0     4.858
  u'sin         5         0.1268            1.0     4.858
    sin         3         0.1268            1.0     4.858
     wa        